# Text Similarity: Within- vs. Cross-Culture
Sentence-BERT embeddings + 10×10 cross-culture similarity matrices.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.analysis.data_loading import load_annotations
from src.analysis.embeddings import EmbeddingManager
from src.analysis.similarity import (
    compute_per_image_similarity, compute_within_cross_similarity,
    compute_culture_sim_matrix, compute_image_conditioned_culture_sim, cluster_order
)
sns.set_theme(style='white')

In [ ]:
df = load_annotations(model_names=['qwen_vl'])  # Start with one model
em = EmbeddingManager(device='mps')

Path('../outputs/analysis').mkdir(parents=True, exist_ok=True)
cap_embs = em.encode_column(df, 'caption', cache_path='../outputs/analysis/caption_embs_qwen_vl.npy')
just_embs = em.encode_column(df, 'justification', cache_path='../outputs/analysis/just_embs_qwen_vl.npy')
print(f'Caption embeddings: {cap_embs.shape}  |  Justification: {just_embs.shape}')

In [ ]:
# Per-image similarity
per_img = compute_per_image_similarity(df, cap_embs)
print(f'Mean caption similarity across culture models: {per_img["sim_mean"].mean():.4f} ± {per_img["sim_mean"].std():.4f}')
per_img.describe()

In [ ]:
# 10×10 image-conditioned culture similarity matrix
ic_matrix, culture_labels = compute_image_conditioned_culture_sim(df, cap_embs)
np.save('../outputs/analysis/ic_culture_sim_qwen_vl.npy', ic_matrix)

order = cluster_order(ic_matrix)
ordered_matrix = ic_matrix[np.ix_(order, order)]
ordered_labels = [culture_labels[i] for i in order]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(ordered_matrix, xticklabels=ordered_labels, yticklabels=ordered_labels,
            annot=True, fmt='.3f', cmap='Reds', ax=ax, vmin=0.5, vmax=1.0)
ax.set_title('Image-Conditioned Cross-Culture Similarity (Caption, Qwen3.5-2B)')
plt.tight_layout()
plt.savefig('../outputs/figures/03_ic_culture_sim_qwen_vl.pdf', bbox_inches='tight')
plt.show()